# Plyphyny Phase Combat Engine
Reference Implementation based on **Eldritch Rules 8.17.2025**

This notebook implements the deterministic **Battle Phase** initiative system. It provides definitions, calculation functions, and a simulation of the sorting logic used to determine turn order in a round.

### Core Mechanics
1. **Battle Phases (1-5)**: Determined by Initiative Score (Prowess MV + Focus).
2. **Pre-Phase 1**: For entities with Prowess d14+.
3. **Follow-Through (Phase 0)**: For runners from the previous Phase 5.
4. **Tie-Breakers**:
   - Priority A: Weapon Reach
   - Priority B: Heroic Tier classification
   - Priority C: Random Roll fallback


In [ ]:
import random
from enum import IntEnum

# Define Enums for Priorities to make comparisons easy
class ReachPriority(IntEnum):
    LONG_RANGE = 1
    LONG_REACH = 2
    MEDIUM_REACH = 3
    SHORT_REACH = 4

class ClassificationPriority(IntEnum):
    PC_LEGENDARY = 1
    EXCEPTIONAL = 2
    STANDARD = 3
    MINOR = 4

class Combatant:
    def __init__(self, name, prowess_die, reaction_focus=0, finesse_focus=0, 
                 reach=ReachPriority.MEDIUM_REACH, 
                 classification=ClassificationPriority.STANDARD,
                 follow_through_active=False):
        self.name = name
        self.prowess_die = prowess_die  # String like 'd4', 'd12', 'd14'
        self.prowess_mv = int(prowess_die[1:]) # Extract number from 'd12' -> 12
        self.reaction_focus = reaction_focus
        self.finesse_focus = finesse_focus
        self.reach = reach
        self.classification = classification
        self.follow_through_active = follow_through_active
        
        # Derived stats
        self.initiative_score = self.prowess_mv + self.reaction_focus + self.finesse_focus
        self.battle_phase = self._calculate_phase()
        
        # Fallback roll for absolute ties (simulating d20 + stats)
        self.fallback_roll = random.randint(1, 20) + self.initiative_score

    def _calculate_phase(self):
        """
        Maps Initiative Score to Phase.
        14+   -> Phase 0 (Pre-Phase 1) - Logic implied by d14+ note
        12+   -> Phase 1
        9-11  -> Phase 2
        7-8   -> Phase 3
        5-6   -> Phase 4
        1-4   -> Phase 5
        """
        s = self.initiative_score
        
        # Special case for "Runners" in Phase 5 -> Act first (Phase 0 effectively)
        # Note: Follow-through overrides calculated phase in sorting, but let's store base phase here.
        
        if s >= 14: return 0  # Represents Pre-Phase 1
        if s >= 12: return 1
        if s >= 9: return 2
        if s >= 7: return 3
        if s >= 5: return 4
        return 5

    def __repr__(self):
        return (f"{self.name} [Phase {self.battle_phase} | Reach {self.reach.name} | "
                f"Class {self.classification.name} | InitScore {self.initiative_score}]")


In [ ]:
def sort_combatants(combatants):
    """
    Sorts a list of combatants based on the Eldritch Rules priority hierarchy.
    """
    return sorted(combatants, key=lambda c: (
        # Priority 0: Follow-Through Mechanic
        # If active, act effectively at Phase -1/0. 
        # Represented by 0 if True, 1 if False to sort True first.
        0 if c.follow_through_active else 1,
        
        # Priority 1: Battle Phase (Ascending 0 -> 5)
        # 0=Pre-Phase 1, 1=Fastest, 5=Slowest
        c.battle_phase,
        
        # Priority 2: Weapon Reach (Ascending 1 -> 4)
        # Long Range acts before Short Reach
        c.reach,
        
        # Priority 3: Classification / Narrative Role (Ascending 1 -> 4)
        # PC/Legendary acts before Minion
        c.classification,
        
        # Priority 4: Random Fallback (Descending)
        # Higher roll goes first
        -c.fallback_roll
    ))

def print_combat_order(sorted_list):
    print(f"{'#':<4} {'Phase':<8} {'Name':<20} {'Reach':<15} {'Class':<12} {'Notes'}")
    print("-" * 80)
    for i, c in enumerate(sorted_list, 1):
        phase_str = f"P{c.battle_phase}"
        
        # Handle special display logic
        notes = []
        if c.follow_through_active:
            phase_str = "FT(P0)"
            notes.append("Follow-Through (-3 Threat)")
        elif c.battle_phase == 0:
            phase_str = "Pre-P1"
            
        print(f"{i:<4} {phase_str:<8} {c.name:<20} {c.reach.name:<15} {c.classification.name:<12} {', '.join(notes)}")


In [ ]:
# Create a diverse roster of combatants based on the examples and rules

roster = [
    # 1. The Lightning-Fast Legend (Pre-Phase 1)
    Combatant("Ancient Dragon", "d16", reaction_focus=4, classification=ClassificationPriority.PC_LEGENDARY),
    
    # 2. Heroic PC with Long Bow (Phase 1/2 borderline)
    # MV 10 + 2 = 12 -> Phase 1. Long Range.
    Combatant("Ranger", "d10", reaction_focus=2, reach=ReachPriority.LONG_RANGE, classification=ClassificationPriority.PC_LEGENDARY),
    
    # 3. Fast Rogue (Phase 2), Short Reach
    # MV 8 + 3 = 11 -> Phase 2.
    Combatant("Rogue", "d8", reaction_focus=3, reach=ReachPriority.SHORT_REACH, classification=ClassificationPriority.PC_LEGENDARY),
    
    # 4. Standard Goblin (Phase 5)
    # MV 4 -> Phase 5.
    Combatant("Goblin Grunt", "d4", classification=ClassificationPriority.MINOR),
    
    # 5. Follow-Through Goblin (Was Slow, but ran previous turn)
    Combatant("Running Goblin", "d4", classification=ClassificationPriority.MINOR, follow_through_active=True),
    
    # 6. Tie-Breaker: Two Fighters in Phase 3
    # Fighter A: Long Reach (Polearm)
    # Fighter B: Medium Reach (Sword)
    Combatant("Polearm Fighter", "d8", reaction_focus=0, reach=ReachPriority.LONG_REACH, classification=ClassificationPriority.STANDARD),
    Combatant("Sword Fighter", "d8", reaction_focus=0, reach=ReachPriority.MEDIUM_REACH, classification=ClassificationPriority.STANDARD),
    
    # 7. Tie-Breaker: PC vs Monster in same phase/reach
    # Both Phase 3 (Score 8), Both Medium Reach
    # PC should win due to Heroic Tier
    Combatant("Hero Knight", "d8", reaction_focus=0, reach=ReachPriority.MEDIUM_REACH, classification=ClassificationPriority.PC_LEGENDARY),
    Combatant("Orc Boss", "d8", reaction_focus=0, reach=ReachPriority.MEDIUM_REACH, classification=ClassificationPriority.EXCEPTIONAL),
]

# Run the Sort
sorted_roster = sort_combatants(roster)

print("Final Combat Order:")
print_combat_order(sorted_roster)


### Tactical Movement Calculation
This section implements the movement algorithm derived from the "Tactical Movement for VTT" supplement. It handles the logic for both Players (Conditional Focus) and Creatures (Automatic Bonus).

#### Logic Summary
1. **Base Formula**: `(12 + Prowess_MV + Agility_MV) / 5`
2. **Rounding**: 
   - Player/NPC with Agility Specialty -> Round UP (`ceil`)
   - Default -> Round DOWN (`floor`)
3. **Bonuses**:
   - **Creature**: Automatic based on BP Die Size.
   - **Player**: Conditional based on Speed Focus rank.
4. **Size Modifier**: Flat addition (e.g., Huge +2).

The following code calculates the speed for the `Huge Basilisk` example.


In [ ]:
import math

class SizeModifier(IntEnum):
    TINY_SMALL = -1
    MEDIUM = 0
    LARGE = 1
    HUGE = 2
    GARGANTUAN = 3

class CreatureType(IntEnum):
    PLAYER = 1   # Speed Focus conditional
    MONSTER = 2  # BP Die automatic bonus

def calculate_movement(prowess_die_str, has_agility_specialty=False, 
                       creature_type=CreatureType.MONSTER, size=SizeModifier.MEDIUM,
                       agility_die_str="d0", speed_focus_rank="d0"):
    """
    Calculates tactical movement in squares per phase.
    """
    # 1. Parse Dice
    prowess_mv = int(prowess_die_str[1:])
    agility_mv = int(agility_die_str[1:]) if has_agility_specialty else 0
    
    # 2. Base Formula
    raw_score = (12 + prowess_mv + agility_mv) / 5.0
    
    # 3. Rounding Rule
    if has_agility_specialty:
        base_squares = math.ceil(raw_score)
        rounding_method = "UP (Agility Spec)"
    else:
        base_squares = math.floor(raw_score)
        rounding_method = "DOWN (No Spec)"
        
    # 4. Tier Bonus (The Fork)
    tier_bonus = 0
    tier_source = ""
    
    if creature_type == CreatureType.MONSTER:
        # Automatic based on BP Die for Monsters
        tier_source = "Automatic (BP Die)"
        if prowess_mv >= 12: tier_bonus = 3
        elif prowess_mv >= 8: tier_bonus = 2
        else: tier_bonus = 1 # d4-d6
    else:
        # Conditional based on Speed Focus for Players
        tier_source = "Conditional (Speed Focus)"
        sf_val = int(speed_focus_rank[1:])
        if sf_val >= 12: tier_bonus = 3
        elif sf_val >= 8: tier_bonus = 2
        elif sf_val >= 4: tier_bonus = 1
        else: tier_bonus = 0
        
    # 5. Size Modifier
    size_mod = size.value
    
    # Total
    total_squares = base_squares + tier_bonus + size_mod
    
    return {
        "Base (Squares)": base_squares,
        "Formula Logic": f"(12 + {prowess_mv} + {agility_mv}) / 5 = {raw_score:.2f} -> {rounding_method}",
        "Tier Bonus": tier_bonus,
        "Tier Source": tier_source,
        "Size Modifier": size_mod,
        "Total Walk": total_squares,
        "Run (x2)": total_squares * 2,
        "Sprint (x4)": total_squares * 4
    }

# --- VALIDATION: HUGE BASILISK EXAMPLE ---
# Stats: d6 BP, No Agility, Huge Size (+2)
basilisk_stats = calculate_movement(
    prowess_die_str="d6", 
    has_agility_specialty=False,
    creature_type=CreatureType.MONSTER,
    size=SizeModifier.HUGE
)

print("--- HUGE BASILISK VALIDATION ---")
for k, v in basilisk_stats.items():
    print(f"{k}: {v}")
